<a href="https://colab.research.google.com/github/paulmunozpauta/Curso_Introduccion_Ciencia_Datos/blob/main/Notebooks/M02_01_Importaci%C3%B3n_exploraci%C3%B3n_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<p align="center">
  <img src="https://github.com/paulmunozpauta/Curso_Introduccion_Ciencia_Datos/raw/main/Static/Imgs/UdeC_color_horizontal.jpg" width="500">
</p>

<p align="center"><b style="font-size:28px;">Facultad de Ingeniería Agrícola</b></p>
<p align="center"><b style="font-size:28px;">Introducción a la Ciencia de Datos</b></p>
<hr>
<p align="center"><b style="font-size:28px;">Contacto</b></p>
<p align="center">
  paulmunoz@udec.cl<br>
  https://paulmunoz.com
</p>

# Introducción a la Ciencia de Datos
## Módulo 2: Organización y preparación de datos

### Notebook 1: Importación y exploración inicial de datos


En este notebook trabajaremos con una serie diaria de precipitación de la estación **Bernardo O'Higgins, Chillán**, obtenida desde el Explorador Climático CR2.

El objetivo es aprender a:

- cargar un archivo de datos en Google Colab;
- importar datos con Python;
- trabajar con un `DataFrame`;
- explorar la estructura de una base de datos;
- preparar una variable de fecha;
- identificar datos faltantes;
- realizar una primera visualización;
- exportar los datos procesados.

## 1. Importar las librerías

En Python, muchas funcionalidades se encuentran organizadas en **librerías**.

En este notebook utilizaremos:

- `pandas` para leer, organizar y procesar datos.
- `matplotlib` para realizar gráficos.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

## 2. Cargar el archivo de datos

Google Colab funciona en la nube. Por esta razón, podemos cargar archivos desde nuestro computador al entorno de trabajo de Colab.

Ejecuta la siguiente celda y selecciona el archivo:

`BernardoOHigginsChillan.xlsx`

In [ ]:
from google.colab import files

uploaded = files.upload()

## 3. Leer el archivo con Pandas

El archivo contiene una hoja llamada `Serie`.

Utilizaremos la función `read_excel()` de Pandas para importar los datos.

In [ ]:
datos = pd.read_excel("BernardoOHigginsChillan.xlsx", sheet_name="Serie")

## 4. ¿Qué es un DataFrame?

Un **DataFrame** es una estructura de datos organizada en filas y columnas.

Es similar a una hoja de cálculo de Excel, pero permite procesar y analizar los datos mediante código.

In [ ]:
datos.head()

### Primeras y últimas observaciones

La función `head()` muestra las primeras filas del DataFrame, mientras que `tail()` permite visualizar las últimas.

In [ ]:
datos.head()

In [ ]:
datos.tail()

## 5. Explorar la estructura de los datos

Antes de realizar cualquier análisis es importante conocer cómo está organizada nuestra base de datos.

Revisaremos:

- número de filas y columnas;
- nombres de las variables;
- tipos de datos;
- información general del DataFrame.

In [ ]:
datos.shape

In [ ]:
datos.columns

In [ ]:
datos.dtypes

In [ ]:
datos.info()

## 6. Seleccionar una columna

Podemos acceder a una variable del DataFrame utilizando el nombre de su columna.

En este caso, `valor` corresponde a la precipitación diaria expresada en milímetros.

In [ ]:
datos["valor"]

In [ ]:
datos["valor"].head(10)

## 7. Crear una variable de fecha

Actualmente el año, mes y día se encuentran almacenados en columnas independientes.

Para analizar una serie de tiempo es más conveniente disponer de una única variable de tipo fecha.

In [ ]:
datos["fecha"] = pd.to_datetime(
    dict(
        year=datos["agno"],
        month=datos["mes"],
        day=datos["dia"]
    )
)

In [ ]:
datos.head()

In [ ]:
datos.dtypes

Para facilitar el trabajo, podemos colocar la fecha al inicio del DataFrame.

In [ ]:
datos = datos[["fecha", "agno", "mes", "dia", "valor"]]

datos.head()

## 8. Renombrar variables

Los datos reales no siempre utilizan nombres de variables fáciles de interpretar.

Podemos cambiar los nombres sin modificar los datos originales.

In [ ]:
datos = datos.rename(columns={
    "agno": "año",
    "valor": "precipitacion_mm"
})

datos.head()

## 9. Identificar datos faltantes

Los datos observados pueden contener registros sin información.

Antes de realizar cualquier análisis es importante identificar estos valores.

In [ ]:
datos.isna().sum()

### Identificar fechas faltantes

Una serie puede no contener valores `NaN` y, aun así, estar incompleta.

Por ejemplo, si después del 30 de noviembre aparece directamente el 2 de diciembre, el 1 de diciembre no está registrado en el DataFrame. Como no existe una fila para esa fecha, `isna()` no puede detectarla.

Para comprobar la continuidad de una serie diaria, podemos generar todas las fechas esperadas entre el inicio y el final de la serie y compararlas con las fechas disponibles.

In [ ]:
# Crear una serie con todas las fechas que deberían existir
fechas_esperadas = pd.date_range(
    start=datos["fecha"].min(),
    end=datos["fecha"].max(),
    freq="D"
)

# Identificar fechas que no están presentes en los datos
fechas_faltantes = fechas_esperadas.difference(datos["fecha"])

print("Número de fechas faltantes:", len(fechas_faltantes))

In [ ]:

fechas_faltantes

In [ ]:
pd.set_option("display.max_rows", None)
fechas_faltantes_df = pd.DataFrame({
    "fecha_faltante": fechas_faltantes
})

fechas_faltantes_df

In [ ]:
pd.reset_option("display.max_rows")

In [ ]:
# Porcentaje de completitud de la serie
completitud = (
    len(datos) / len(fechas_esperadas)
) * 100

print(f"Completitud de la serie: {completitud:.2f}%")

## 10. Una primera descripción de los datos

Pandas permite obtener rápidamente algunas características generales de las variables numéricas.

In [ ]:
datos["precipitacion_mm"].describe()

In [ ]:
datos["precipitacion_mm"].max()

In [ ]:
datos.loc[datos["precipitacion_mm"].idxmax()]

## 11. Primera visualización de la serie

Una serie temporal permite observar cómo cambia una variable a través del tiempo.

Por ahora realizaremos una visualización simple. En el siguiente módulo aprenderemos a construir e interpretar gráficos con mayor detalle.

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    datos["fecha"],
    datos["precipitacion_mm"]
)

plt.xlabel("Fecha")
plt.ylabel("Precipitación diaria (mm)")
plt.title("Precipitación diaria - Estación Bernardo O'Higgins, Chillán")

plt.show()

### Visualizar un periodo específico

También podemos seleccionar solamente una parte de la serie.

A continuación observaremos la precipitación durante el año 2025.

In [ ]:
datos_2025 = datos[datos["año"] == 2025]

datos_2025

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    datos_2025["fecha"],
    datos_2025["precipitacion_mm"]
)

plt.xlabel("Fecha")
plt.ylabel("Precipitación diaria (mm)")
plt.title("Precipitación diaria en Chillán - 2025")

plt.show()

## 12. Exportar los datos procesados

Después de modificar o procesar una base de datos podemos guardar el resultado en un nuevo archivo.

Utilizaremos el formato CSV (`Comma-Separated Values`), ampliamente utilizado para almacenar datos tabulares.

In [ ]:
datos.to_csv(
    "precipitacion_chillan_procesada.csv",
    index=False
)

In [ ]:
files.download("precipitacion_chillan_procesada.csv")

# Actividad práctica

Ahora debes reproducir el procedimiento utilizando la serie de precipitación de **otra estación** obtenida desde el Explorador Climático CR2.


1. Carga el archivo en Google Colab.
2. Importa la hoja que contiene la serie de datos.
3. Guarda los datos en un DataFrame.
4. Muestra las primeras y últimas filas.
5. Determina el número de filas y columnas.
6. Identifica los nombres y tipos de las variables.
7. Construye una columna de fecha.
8. Renombra las variables para que sean fáciles de interpretar.
9. Determina si existen datos faltantes.
10. Obtén el valor máximo de precipitación.
11. Identifica la fecha en que ocurrió.
12. Realiza un gráfico de la serie completa.
13. Selecciona un año y realiza un segundo gráfico.
14. Exporta el DataFrame procesado como archivo CSV.